In [ ]:
# =====================================================================
# MODEL: Artificial Neural Network (ANN) - Multilayer Perceptron
# DATASET: combined_outages_compressed.csv
# GOAL: Predict 'TIME_DURATION' in minutes (Regression)
# =====================================================================
import pandas as pd
import numpy as np
import re
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping

def parse_duration(time_str):
    if pd.isna(time_str): return 0.0
    time_str = str(time_str).upper()
    hours, mins = 0, 0
    h_match = re.search(r'(\d+)\s*H', time_str)
    m_match = re.search(r'(\d+)\s*M', time_str)
    if h_match: hours = int(h_match.group(1))
    if m_match: mins = int(m_match.group(1))
    return float(hours * 60 + mins)

# 1. Load Data
df = pd.read_csv('combined_outages_compressed.csv')
df['DURATION_MINS'] = df['TIME_DURATION'].apply(parse_duration)
df = df[df['DURATION_MINS'] > 0]

cols_to_drop = ['TIME_DURATION', 'DURATION_MINS', 'ENTRYDATE', 'DOWN_DATE', 'DOWN_INFO']
X = df.drop(columns=[c for c in cols_to_drop if c in df.columns])
y = df['DURATION_MINS'].values # Array format for Keras

# 2. Preprocessing
cat_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()
num_cols = X.select_dtypes(include=['number']).columns.tolist()

preprocessor = ColumnTransformer(transformers=[
    ("num", Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())]), num_cols),
    ("cat", Pipeline([('imputer', SimpleImputer(strategy='constant', fill_value='missing')), ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))]), cat_cols)
])

# 3. Split and Transform
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train_prep = preprocessor.fit_transform(X_train)
X_test_prep = preprocessor.transform(X_test)

# 4. Build ANN (Regression)
ann_model = Sequential([
    Dense(256, activation="relu", input_shape=(X_train_prep.shape[1],)),
    BatchNormalization(), Dropout(0.3),
    Dense(128, activation="relu"),
    BatchNormalization(), Dropout(0.3),
    Dense(64, activation="relu"),
    Dense(1) # Linear activation for continuous output
])

ann_model.compile(optimizer='adam', loss='mse', metrics=['mae'])
early_stopping = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

# 5. Train
print("Training ANN Regressor on Combined Outages...")
ann_model.fit(X_train_prep, y_train, epochs=30, batch_size=64, validation_split=0.2, callbacks=[early_stopping], verbose=1)

# 6. Evaluate
y_pred = ann_model.predict(X_test_prep).flatten()

print("\n--- ANN RESULTS (DURATION REGRESSION) ---")
print("R2 Score:", r2_score(y_test, y_pred))
print("RMSE (Minutes):", np.sqrt(mean_squared_error(y_test, y_pred)))
print("MAE (Minutes):", mean_absolute_error(y_test, y_pred))